# Whisper (Small) — Toy Encoder-Decoder ASR Transformer

In [ ]:
import mathimport torchimport torch.nn as nnimport torch.nn.functional as Ffrom torch.utils.data import Dataset, DataLoader

In [ ]:
class Config:    n_mels = 80    audio_len = 200    vocab_size = 300    max_text_len = 40    d_model = 128    n_heads = 4    n_encoder_layers = 3    n_decoder_layers = 3    dropout = 0.1    pad_id = 0    bos_id = 1    eos_id = 2cfg = Config()device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
class SinusoidalPositionalEncoding(nn.Module):    def __init__(self, d_model, max_len):        super().__init__()        pe = torch.zeros(max_len, d_model)        position = torch.arange(0, max_len).unsqueeze(1).float()        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))        pe[:, 0::2] = torch.sin(position * div_term)        pe[:, 1::2] = torch.cos(position * div_term)        self.register_buffer("pe", pe.unsqueeze(0))    def forward(self, x):        return x + self.pe[:, :x.size(1)]

In [ ]:
class AudioEncoderStem(nn.Module):    def __init__(self, cfg):        super().__init__()        self.conv1 = nn.Conv1d(cfg.n_mels, cfg.d_model, kernel_size=3, padding=1)        self.conv2 = nn.Conv1d(cfg.d_model, cfg.d_model, kernel_size=3, stride=2, padding=1)        self.pos_enc = SinusoidalPositionalEncoding(cfg.d_model, cfg.audio_len)    def forward(self, mel):        x = mel.transpose(1, 2)        x = F.gelu(self.conv1(x))        x = F.gelu(self.conv2(x))        x = x.transpose(1, 2)        return self.pos_enc(x)

In [ ]:
class WhisperEncoder(nn.Module):    def __init__(self, cfg):        super().__init__()        self.stem = AudioEncoderStem(cfg)        layer = nn.TransformerEncoderLayer(            d_model=cfg.d_model, nhead=cfg.n_heads, dim_feedforward=4 * cfg.d_model,            dropout=cfg.dropout, batch_first=True        )        self.layers = nn.TransformerEncoder(layer, num_layers=cfg.n_encoder_layers)        self.ln_post = nn.LayerNorm(cfg.d_model)    def forward(self, mel):        x = self.stem(mel)        x = self.layers(x)        return self.ln_post(x)

In [ ]:
class WhisperDecoder(nn.Module):    def __init__(self, cfg):        super().__init__()        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model, padding_idx=cfg.pad_id)        self.pos_emb = nn.Parameter(torch.zeros(1, cfg.max_text_len, cfg.d_model))        layer = nn.TransformerDecoderLayer(            d_model=cfg.d_model, nhead=cfg.n_heads, dim_feedforward=4 * cfg.d_model,            dropout=cfg.dropout, batch_first=True        )        self.layers = nn.TransformerDecoder(layer, num_layers=cfg.n_decoder_layers)        self.ln_f = nn.LayerNorm(cfg.d_model)        self.head = nn.Linear(cfg.d_model, cfg.vocab_size)    def forward(self, tokens, memory):        t = tokens.size(1)        x = self.tok_emb(tokens) + self.pos_emb[:, :t]        causal_mask = torch.triu(torch.ones(t, t, device=tokens.device), diagonal=1).bool()        pad_mask = tokens == 0        x = self.layers(x, memory, tgt_mask=causal_mask, tgt_key_padding_mask=pad_mask)        x = self.ln_f(x)        return self.head(x)

In [ ]:
class Whisper(nn.Module):    def __init__(self, cfg):        super().__init__()        self.encoder = WhisperEncoder(cfg)        self.decoder = WhisperDecoder(cfg)    def forward(self, mel, tokens):        memory = self.encoder(mel)        logits = self.decoder(tokens, memory)        return logits    def greedy_decode(self, mel, max_len, bos_id, eos_id):        memory = self.encoder(mel)        generated = torch.full((mel.size(0), 1), bos_id, dtype=torch.long, device=mel.device)        for _ in range(max_len - 1):            logits = self.decoder(generated, memory)            next_token = logits[:, -1].argmax(dim=-1, keepdim=True)            generated = torch.cat([generated, next_token], dim=1)            if (next_token == eos_id).all():                break        return generated

In [ ]:
class ToyASRDataset(Dataset):    def __init__(self, n_samples, cfg):        self.n_samples = n_samples        self.cfg = cfg    def __len__(self):        return self.n_samples    def __getitem__(self, idx):        mel = torch.randn(self.cfg.audio_len, self.cfg.n_mels)        text_len = torch.randint(5, self.cfg.max_text_len - 2, (1,)).item()        tokens = torch.randint(3, self.cfg.vocab_size, (text_len,))        tokens = torch.cat([            torch.tensor([self.cfg.bos_id]),            tokens,            torch.tensor([self.cfg.eos_id])        ])        return mel, tokensdef collate_fn(batch):    mels, tokens = zip(*batch)    mel_batch = torch.stack(mels)    max_len = max(t.size(0) for t in tokens)    token_batch = torch.zeros(len(batch), max_len, dtype=torch.long)    for i, t in enumerate(tokens):        token_batch[i, :t.size(0)] = t    return mel_batch, token_batch

In [ ]:
dataset = ToyASRDataset(n_samples=64, cfg=cfg)loader = DataLoader(dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)model = Whisper(cfg).to(device)optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

In [ ]:
def train_epoch(model, loader, optimizer):    model.train()    total_loss = 0.0    for mel, tokens in loader:        mel = mel.to(device)        tokens = tokens.to(device)        decoder_input = tokens[:, :-1]        target = tokens[:, 1:]        logits = model(mel, decoder_input)        loss = F.cross_entropy(logits.reshape(-1, cfg.vocab_size), target.reshape(-1), ignore_index=cfg.pad_id)        optimizer.zero_grad()        loss.backward()        optimizer.step()        total_loss += loss.item()    return total_loss / len(loader)

In [ ]:
for epoch in range(5):    avg_loss = train_epoch(model, loader, optimizer)    print(f"epoch {epoch + 1} loss {avg_loss:.4f}")

In [ ]:
model.eval()with torch.no_grad():    mel_sample = torch.randn(1, cfg.audio_len, cfg.n_mels).to(device)    transcript_tokens = model.greedy_decode(mel_sample, max_len=cfg.max_text_len, bos_id=cfg.bos_id, eos_id=cfg.eos_id)    print(transcript_tokens)